In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục src từ: {cwd}")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root}")

Project root: /home/trieu/Intern/SwM_precomputed


In [2]:
from src.data.train_dataset import TrainDataset
from torch.utils.data import DataLoader
from src.utils.io import load_pickle

In [3]:
train_samples = [
    {
        'history': [
            'N8129',
            'N1569',
            'N17686',
        ],
        'target': 'N13008',
    },
    {
        'history': [
            'N63302',
            'N10414',
            'N19347',
            'N31801'
        ],
        'target': 'N55689'
    },
    {
        'history': [
            'N21623',
            'N6233',
            'N14340',
            'N48031',
            'N62285'
        ],
        'target': 'N31739'
    },
    {
        'history': [
            'N31739',
            'N6072',
            'N63045',
            'N23979',
            'N35656',
        ],
        'target': 'N43353'
    },
    
]

In [4]:
mapping = load_pickle(project_root / "data/processed/mindsmall_v2/artifacts/news_vector_mapping.pkl")
train_dataset = TrainDataset(samples=train_samples, max_sequence_length=5, padding_id=0, mapping=mapping, vector_size=384)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [5]:
from src.models.sasrec import SASRec
import torch

In [6]:
model = SASRec(
    max_sequence_length=20,
    num_blocks=2,
    num_heads=2,
    dropout=0.1,
    embedding_dim=384
)

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
model.train()
total_loss = 0.0
batch = next(iter(train_loader))

/home/trieu/Intern/SwM_precomputed/src/data/train_dataset.py:75: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  "input_vectors": torch.tensor(


In [9]:
input_vectors = batch['input_vectors']
positive_vectors = batch['positive_vectors']
negative_vectors = batch['negative_vectors']
optimizer.zero_grad()
outputs = model(
    input_vectors=input_vectors,
    positive_vectors=positive_vectors,
    negative_vectors=negative_vectors,
)
outputs

{'positive_logits': tensor([[ 0.0000, -0.4937,  1.6493, -0.2918,  1.7893],
         [-0.1045, -0.3785,  0.6313, -0.0150, -0.2674]], grad_fn=<SumBackward1>),
 'negative_logits': tensor([[ 0.0000, -0.0720, -0.4692,  1.1499,  1.1140],
         [-0.4148, -0.8445,  0.5330,  1.9890,  0.4680]], grad_fn=<SumBackward1>),
 'sequence_output': tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [-1.1518,  0.4652, -1.7150,  ...,  0.4866,  0.9391, -0.2387],
          [ 1.5356, -0.9444,  0.6942,  ...,  0.7049,  0.0203,  0.1782],
          [-1.1755, -0.7909,  0.3600,  ...,  0.4650, -1.9432,  0.4273],
          [-1.3060, -0.4720, -0.7806,  ...,  0.5804, -1.1935, -0.3342]],
 
         [[ 0.8881, -0.0669,  0.8525,  ..., -0.2922, -1.3097, -1.1047],
          [-1.0291,  0.2707, -2.2357,  ..., -0.3020,  1.7982, -0.4504],
          [ 1.0953, -0.7650,  0.8993,  ...,  0.2744, -0.3849,  0.4806],
          [-1.4975,  0.4120, -1.4225,  ...,  0.2452, -1.4821,  0.3202],
          [-1.79

In [18]:
input_vectors.shape, positive_vectors.shape, negative_vectors.shape

(torch.Size([2, 5, 384]), torch.Size([2, 5, 384]), torch.Size([2, 5, 384]))

In [19]:
positive_vectors.shape[:-1]

torch.Size([2, 5])

In [10]:
positive_logits = outputs['positive_logits']
negative_logits = outputs['negative_logits']

In [11]:
positive_labels = torch.ones_like(positive_logits)
negative_labels = torch.zeros_like(negative_logits)
positive_labels, negative_labels

(tensor([[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

In [12]:
loss_fn = torch.nn.BCEWithLogitsLoss()

In [13]:
positive_loss = loss_fn(positive_logits, positive_labels)
negative_loss = loss_fn(negative_logits, negative_labels)
positive_loss, negative_loss

(tensor(0.6453, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>),
 tensor(0.9591, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>))

In [14]:
if positive_logits.shape != negative_logits.shape:
    raise ValueError("positive_logits and negative_logits must have equal shape")
if positive_vectors.ndim != positive_logits.ndim + 1:
    raise ValueError("positive_vectors must have shape [B, L, T]")
if positive_vectors.shape[:-1] != positive_logits.shape:
    raise ValueError("Token batch/sequence dimensions must match logits")

In [20]:
valid_mask = positive_vectors.ne(0).any(dim=-1).to(positive_loss.dtype)
valid_mask

tensor([[0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [ ]:

loss = (positive_loss + negative_loss) * valid_mask
loss = loss.sum() / valid_mask.sum().clamp_min(1.0)

In [17]:
loss

tensor(1.6044, grad_fn=<DivBackward0>)